In [1]:
!pip install langchain-text-splitters
!pip install langchain-openai
!pip install langchain_classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 23.7 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.6 MB/s eta 0:00:00


In [3]:
import os
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from dotenv import load_dotenv

from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

llm = ChatOpenAI(model="gpt-4o-mini", api_key=api_key)

In [ ]:
# json, table (markdown), yaml, xml

In [4]:
plain_prompt = """다음 제품 리뷰를 분석해줘. 전체 감정(긍정/부정/혼합), 1~5점 점수, 장점 목록, 단점 목록, 핵심 키워드 3개를 알려줘.
리뷰 : "이 노트북 정말 가벼워서 좋아요! 다만 키보드 타건감이 아쉽네요"
유효한 json만 출력하세요"""

print(llm.invoke([HumanMessage(content=plain_prompt)]).content)

```json
{
  "전체 감정": "혼합",
  "점수": 4,
  "장점": [
    "가벼움"
  ],
  "단점": [
    "키보드 타건감 아쉬움"
  ],
  "핵심 키워드": [
    "가벼움",
    "키보드",
    "타건감"
  ]
}
```


In [5]:
markdown_prompt = """# 제품 리뷰 감정 분석

## 입력 리뷰
> "이 노트북 정말 가벼워서 좋아요! 다만 키보드 타건감이 아쉽네요"

## 분석 항목
- **overall_sentiment** : 긍정/부정/혼합 중 하나
- **score** : 1~5점수
- **pros** : 장점 리스트
- **cons** : 단점 리스트
- **keywords** : 핵심 키워드 3개

## 출력 형식
유효한 json만 출력하세요. 다른 텍스트를 포함하지 마세요
"""

print(llm.invoke([HumanMessage(content=markdown_prompt)]).content)

{
  "overall_sentiment": "혼합",
  "score": 4,
  "pros": [
    "가벼운 무게"
  ],
  "cons": [
    "타건감 아쉬움"
  ],
  "keywords": [
    "노트북",
    "가벼움",
    "키보드"
  ]
}


In [6]:
json_prompt = """다음 제품 리뷰를 분석해서 json 형식으로 출력하세요

리뷰 : "이 노트북 정말 가벼워서 좋아요! 다만 키보드 타건감이 아쉽네요"

출력 형식
{
    "overall_sentiment" : 긍정/부정/혼합,
    "score" : 1~5점수,
    "pros" : [장점],
    "cons" : [단점],
    "keywords" : [키워드1, 키워드2, 키워드3]
}
"""

print(llm.invoke([HumanMessage(content=json_prompt)]).content)

```json
{
    "overall_sentiment": "혼합",
    "score": 3,
    "pros": ["가벼움"],
    "cons": ["키보드 타건감"],
    "keywords": ["노트북", "가벼움", "키보드"]
}
```


In [ ]:
# Parser

In [16]:
# 회의록 -> 구조화된 데이터로 변환
def generate_meeting_minute(raw_text, json_structure):
    messages = [
        SystemMessage(content = '당신은 회의록 전문가입니다. 부연설명 없이 회의록을 json데이터만 반환하세요.'),
        HumanMessage(content = f'다음 회의 내용을 지정된 구조로 정리해 주세요.\n {json_structure} \n\n회의내용: {raw_text}')
    ]

    return llm.invoke(messages).content

In [18]:
json_structure = {
    'date' : 'YYYY-MM-DD',
    'attendees' : ['이름1', '이름2'],
    'agenda' : ['안건1', '안건2'],
    'decisions' : ['결정1'],
    'action_items' : [{'assignee' : '담당자', 'task' : '작업', 'deadline' : '기한'}]
}

raw_text="""
3월 15일 마케팅팀 주간 회의. 참석: 김팀장, 이대리, 박사원. 신규 SNS 캠페인 예산 5000만원 확정. 이대리가 3월 22일까지 시안 준비. 박사원은 경쟁사 분석 보고서 3월 20일까지.
"""
response = generate_meeting_minute(raw_text, json_structure)
print(response)

```json
{
  "date": "2023-03-15",
  "attendees": ["김팀장", "이대리", "박사원"],
  "agenda": ["신규 SNS 캠페인 예산 확정", "경쟁사 분석 보고서"],
  "decisions": ["신규 SNS 캠페인 예산 5000만원 확정"],
  "action_items": [
    {
      "assignee": "이대리",
      "task": "시안 준비",
      "deadline": "2023-03-22"
    },
    {
      "assignee": "박사원",
      "task": "경쟁사 분석 보고서",
      "deadline": "2023-03-20"
    }
  ]
}
```


In [ ]:
# prompt: 제약조건
# 3문장로 요약해서 말하세요 (길이 제한),
# 검색된 정보로만 말하세요 (RAG, 정보 제한),
# Json 형태로만 출력하세요 (포맷 제한)
# 부정 제약보다 긍정 제약이 좀 더 정확한 결과를 도출한다.
# .... 제약 규격 통일

In [24]:
class ConstrainedPrompt:
  def __init__(self, base_instruction):
    self.instruction = base_instruction
    self.constraints = []

  def add_length(self, description):
    self.constraints.append(f'[길이] {description}')
    return self

  def add_content(self, description):
    self.constraints.append(f'[내용] {description}')
    return self

  def add_format(self, description):
    self.constraints.append(f'[포맷] {description}')
    return self

  def add_style(self, description):
    self.constraints.append(f'[스타일] {description}')
    return self

  def build(self):
    parts = [self.instruction, '\n제약조건:']
    for c in self.constraints:
      parts.append(f' - {c}')
    return '\n'.join(parts)

  def execute(self, **kwargs):
    llm = ChatOpenAI(model="gpt-4o-mini", api_key=api_key, temperature=kwargs['temperature'], max_tokens=kwargs['max_tokens'])
    prompt = self.build()
    return llm.invoke([HumanMessage(content=prompt)]).content

In [26]:
result = (
    ConstrainedPrompt('클라우드 컴퓨팅의 장점을 설명해 주세요.')
      .add_length('다섯개의 불릿 포인트')
      .add_content('비용, 확장성, 보안 관점을 반드시 포함할 것')
      .add_format('각 포인트는 한 줄로, 이모지로 시작할 것')
      .add_style('IT 비전공 경영진을 대상으로 전문 용어에 괄호로 설명을 추가')
      .execute(temperature=0.3,max_tokens=1000)
)

In [23]:
print(result)

- 💰 **비용 절감**: 초기 IT 투자 비용이 낮고, 필요에 따라 사용한 만큼만 지불하는 모델로 운영 비용을 줄일 수 있습니다.  
- 📈 **확장성**: 비즈니스 성장에 따라 즉각적으로 리소스(자원)를 추가하거나 축소할 수 있어 유연한 운영이 가능합니다.  
- 🔒 **보안 강화**: 전문 클라우드 서비스 제공자는 최신 보안 기술을 적용하여 데이터를 안전하게 보호하는 데 도움을 줍니다.  
- 🛠️ **관리 용이성**: IT 인프라의 유지 관리 부담을 줄이고, 인력과 시간을 다른 핵심 비즈니스에 더 집중할 수 있습니다.  
- 🌍 **접근성**: 인터넷만 있으면 언제 어디서나 데이터와 애플리케이션에 접근할 수 있어 유연한 근무 환경을 지원합니다.  


In [35]:
# ConstrainedPrompt을 이용해서 광고카피 / 상세설명을 출력하세요.
product = "에어프로 맥스 무선 헤드폰"
features = ["40시간 배터리", "멀티포인트 연결", "30dB 노이즈캔슬링", "300g 경량 설계"]

response = (
    ConstrainedPrompt(f'{product} 광고카피/상세설명을 작성해 주세요. 특징: {features}')
      .add_length('광고카피는 20자 내외로 다섯개의 리스트로 출력해 줘. \n상세설명은 최대 두 문장, 200자 이내로 간결하게 작성해 줘.')
      .add_content('특징을 반드시 포함할 것')
      .add_format('슬로건 형태로 마침표 없이')
      .add_style('2030 타겟, 감성적이고 트렌디한 톤')
      .execute(temperature=1, max_tokens=500)
)
print(response)

### 광고카피
1. 40시간 음악의 자유  
2. 두 기기 동시 연결 가능  
3. 소음 없는 편안함 경험  
4. 가벼운 디자인으로 언제나 함께  
5. 음악에 몰입하는 순간을  

### 상세설명  
에어프로 맥스 무선 헤드폰은 40시간의 강력한 배터리로 끊김 없는 음악을 제공합니다. 30dB 노이즈캔슬링과 300g의 초경량 설계로 편안한 착용감을 자랑하며, 멀티포인트 연결로 두 기기를 동시에 사용 가능합니다.


In [ ]:
# 컨텍스트 제공
# RAG: 제공 문맥 최 앞단, 뒷단의 내용을 중요하게 읽는다는 논문..
# https://arxiv.org/pdf/2307.03172

In [36]:
# 컨텍스트를 제공
company_policy = """
[모두컴퍼니 재택근무 정책 v2.3]
- 주 3일 재택, 2일 출근 (화/목 필수 출근)
- 재택근무 시 오전 9시까지 Slack 상태 '업무중' 설정 필수
- 해외 원격근무는 최대 연속 2주까지 가능 (사전 승인 필요)
- 야간근무(22시 이후) 시 익일 오후 출근 가능
- 재택근무 장비 지원금: 연 100만원 (영수증 제출)
"""

In [37]:
question = '해외에서 한 달 동안 원격 근무를 할 수 있나요?'
with_context = f'''
아래 회사 정책 문서를 참고하여 질문에 답하세요.
문서에 없는 내용은 \"해당 정책 문서에 명시되어 있지 않습니다\"라고 답하세요. ㅓ

정책문서:
\"\"\"{company_policy}\"\"\"
질문: {question}
'''
print(llm.invoke([HumanMessage(content=with_context)]).content)

해당 정책 문서에 명시되어 있지 않습니다.


In [57]:
def answer_with_context(context, question):
  clause = """
    중요: 반드시 제공된 문서의 내용만을 근거로 답하세요.
    문서에 없는 내용은 \"해당 정책 문서에 명시되어 있지 않습니다\"라고 답하세요.
    추측하거나 외부 지식을 사용하지 마세요.
  """

  prompt = f"""
    아래 참고 문서를 기반으로 질문에 답하세요.
    {clause}

    참고 문서:
    \'\'\'{context}\'\'\'

    질문: {question}

    답변 형식:
    - 답변: [핵심 답변]
    - 근거: [문서에서 관련된 부분 인용]
  """
  return llm.invoke([HumanMessage(content=prompt)]).content

In [59]:
question = '해외에서 1년 동안 원격 근무를 할 수 있나요?'
result = answer_with_context(company_policy, question)
print(result)

- 답변: 아니요, 해외에서 1년 동안 원격 근무를 할 수 없습니다.
- 근거: "해외 원격근무는 최대 연속 2주까지 가능 (사전 승인 필요)"


In [ ]:
# transformer에 대한 직관적인 시각화 사이트
# https://poloclub.github.io/transformer-explainer/
# transformer이 뭔지? -> 번역 input(incoder) - output(decoder)
# encoding --> input 단어들의 관계(transformer 과정)
# decoding --> output

In [ ]:
# zero-shot, few-shot : 예시 제공
# 식대 지원금에 대한 내용이 포함되어 있지 않습니다. -> 감정 분석해 주세요.
# 식대 지원금에 대한 내용이 포함되어 있지 않습니다.:감정 -> 제공된 문서에 해당하는 정보가 없습니다. 감정 분석해 주세요.

In [61]:
# zero-shot: 아무 예시 없이 response를 기대함
categories = ["기술", "경제", "스포츠", "문화", "정치"]
news_articles = [
    "삼성전자가 차세대 AI 반도체 개발에 3조원을 투자한다고 발표했다.",
    "한국은행이 기준금리를 0.25%p 인하하며 경기 부양에 나섰다.",
    "손흥민이 프리미어리그 시즌 최다 도움을 기록하며 팀 승리를 이끌었다.",
    "국립현대미술관에서 한국 현대미술 50년 특별전이 개막했다."
]

In [62]:
classify_prompt = f'''
  당신은 뉴스 분류 전문가입니다. 주어진 뉴스 기사를 다음 카테고리 중 하나로 분류하세요.
  반드시 아래 카테고리 이름만 출력하세요.

  카테고리: {'. '.join(categories)}
'''
for article in news_articles:
  result = llm.invoke([
      SystemMessage(content=classify_prompt),
      HumanMessage(content=f'{classify_prompt}\n뉴스 기사: {article}')
  ]).content
  print(f' 기사: {article[:30]}...')
  print(f' 분류: {result}\n')

 기사: 삼성전자가 차세대 AI 반도체 개발에 3조원을 투자한다...
 분류: 기술

 기사: 한국은행이 기준금리를 0.25%p 인하하며 경기 부양에...
 분류: 경제

 기사: 손흥민이 프리미어리그 시즌 최다 도움을 기록하며 팀 승...
 분류: 스포츠

 기사: 국립현대미술관에서 한국 현대미술 50년 특별전이 개막했...
 분류: 문화



In [65]:
# few-shot: 예시를 통해서 출력 포맷을 정형화 (one-shot, few-shot...)
# 토큰 비용이 많이 듬.
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

few_shot_messages = [
    SystemMessage(content = '주어진 리뷰의 감정을 분석하세요'),
    # 예시
    HumanMessage(content ="리뷰: '이 제품 정말 최고입니다! 강력 추천해요'"),
    AIMessage(content = '{"sentiment" : "긍정", "score" : 0.95, "keywords" : ["최고", "강력 추천"]}'),
    # 예시
    HumanMessage(content = "리뷰: '배송도 느리고 제품 품질도 형편없네요!'"),
    AIMessage(content = '{"sentiment" : "부정", "score" : 0.15, "keywords" : ["느리고", "형편없네요"]}'),
    # 예시
    HumanMessage(content ="리뷰: '가격 대비 그럭저럭 쓸만하네요'"),
    AIMessage(content = '{"sentiment" : "혼합", "score" : 0.6, "keywords" : ["가격 대비", "쓸만함"]}'),

    # 실제 쿼리(질의)
    HumanMessage(content ="리뷰: '디자인은 예쁜데, 배터리가 빨리 닳아요. 전체적으로 보통입니다.'"),
]

print(llm.invoke(few_shot_messages).content)

{"sentiment" : "혼합", "score" : 0.5, "keywords" : ["예쁜 디자인", "배터리 빨리 닳음", "보통"]}


In [66]:
# few shot 실습
# examples처럼 test_message가 출력될 수 있도록 llm에 요청
examples = [
  {"informal": "내일 미팅 좀 미룰 수 있을까? 갑자기 일이 생겼어.",
    "formal": "안녕하세요. 내일 예정된 미팅 일정 변경을 요청드립니다. 긴급한 업무가 발생하여 조율이 필요합니다. 가능한 대체 일정을 알려주시면 감사하겠습니다."},
  {"informal": "그 보고서 다 했어? 빨리 보내줘.",
    "formal": "안녕하세요. 요청드렸던 보고서 진행 상황을 확인드립니다. 완료되셨다면 전달 부탁드리며, 추가 시간이 필요하시면 말씀해 주세요."},
  {"informal": "이번 프로젝트 결과 별로인데 어떻게 할까?",
    "formal": "안녕하세요. 이번 프로젝트 결과에 대해 논의가 필요합니다. 개선 방안을 함께 검토하기 위해 미팅을 잡는 것이 어떨까요?"}
]
test_messages = [
  "다음 주 워크숍 참석 못 할 것 같아. 다른 사람 보내도 돼?",
  "예산 좀 더 받을 수 있을까? 지금 부족해."
]

messages = [
    SystemMessage(content="당신은 비즈니스 커뮤니케이션 전문가입니다. 비격식적인 문장을 격식있는 문장으로 바꾸어주세요.")
]

for example in examples:
    messages.append(HumanMessage(content=example["informal"]))
    messages.append(AIMessage(content=example["formal"]))

for test_msg in test_messages:
    messages.append(HumanMessage(content=test_msg))

print(llm.invoke(messages).content)

안녕하세요. 다음 주 워크숍에 참석할 수 없을 것 같습니다. 대신 다른 분이 참석해도 괜찮은지 확인해 주실 수 있을까요? 

더불어, 현재 예산이 부족하여 추가 예산을 요청드리고자 합니다. 가능 여부를 검토해 주시면 감사하겠습니다.
